### Import Library

In [1]:
import pandas as pd
import re

### Load Dataset

In [2]:
df = pd.read_csv('../data/dataMentah.csv')
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80472 entries, 0 to 80471
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Judul    80472 non-null  object
 1   Waktu    80472 non-null  object
 2   Link     80472 non-null  object
 3   Content  80463 non-null  object
 4   tag1     78023 non-null  object
 5   tag2     77612 non-null  object
 6   tag3     71602 non-null  object
 7   tag4     56856 non-null  object
 8   tag5     35633 non-null  object
 9   source   80472 non-null  object
dtypes: object(10)
memory usage: 6.1+ MB


In [4]:
df.shape

(80472, 10)

In [5]:
# Cek tag 20 teratas pada tag1
print(df['tag1'].value_counts().head(20))

tag1
Jakarta             865
prabowo subianto    774
kpk                 743
prabowo             635
Prabowo             541
KPK                 501
Pramono Anung       415
Ridwan Kamil        340
kebakaran           339
jokowi              327
israel              311
info-tempo          305
mpr                 260
banjir              258
gempa               253
pramono anung       244
kecelakaan          239
PDI-P               235
Prabowo Subianto    222
mbg                 221
Name: count, dtype: int64


In [6]:
# Cek tag 20 teratas pada tag2
print(df['tag2'].value_counts().head(20))

tag2
jabodetabek         488
prabowo subianto    471
info-tempo          423
prabowo             417
Pilkada Jakarta     352
israel              290
korupsi             290
kpk                 258
KPK                 240
Prabowo             232
Pramono Anung       217
Jokowi              216
pilkada 2024        216
bogor               198
jokowi              195
bmkg                191
imsakiyah           180
pilkada             175
Prabowo Subianto    175
dpr                 167
Name: count, dtype: int64


#### Mapping

In [7]:
def map_label(row):
    # Gabungkan tag1 dan tag2 untuk pengecekan lebih akurat
    tags = f"{str(row['tag1']).lower()} {str(row['tag2']).lower()}"
    
    if any(x in tags for x in ['prabowo', 'jokowi', 'pramono', 'ridwan kamil', 'mpr', 'dpr', 'pdi-p', 'pilkada', 'ldp']):
        return 'Politik'
    elif any(x in tags for x in ['kpk', 'korupsi', 'polisi', 'polres', 'polda', 'hukum', 'sidang']):
        return 'Hukum_Kriminal'
    elif any(x in tags for x in ['kebakaran', 'banjir', 'gempa', 'bmkg', 'kecelakaan', 'bencana']):
        return 'Bencana'
    elif any(x in tags for x in ['israel', 'internasional', 'luar negeri', 'pm']):
        return 'Internasional'
    elif any(x in tags for x in ['jakarta', 'jabodetabek', 'bogor', 'nasional']):
        return 'Nasional'
    else:
        return 'Lainnya'

df['category'] = df.apply(map_label, axis=1)

# Cek hasil distribusi kategori baru
print(df['category'].value_counts())

category
Lainnya           53719
Politik           12303
Hukum_Kriminal     5550
Nasional           4323
Bencana            3380
Internasional      1197
Name: count, dtype: int64


In [8]:
# Menghapus kategori Lainnya agar model fokus pada kategori yang jelas
df = df[df['category'] != 'Lainnya'].reset_index(drop=True)

# Cek distribusi baru
print(df['category'].value_counts())

category
Politik           12303
Hukum_Kriminal     5550
Nasional           4323
Bencana            3380
Internasional      1197
Name: count, dtype: int64


In [9]:
# Menyeimbangkan data agar model tidak bias
df_balanced = df.groupby('category').apply(lambda x: x.sample(n=min(len(x), 4000), random_state=42)).reset_index(drop=True)

print(df_balanced['category'].value_counts())

category
Hukum_Kriminal    4000
Politik           4000
Nasional          4000
Bencana           3380
Internasional     1197
Name: count, dtype: int64


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4424\1695827830.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('category').apply(lambda x: x.sample(n=min(len(x), 4000), random_state=42)).reset_index(drop=True)


#### PreProcesing

In [10]:
pip install swifter

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import swifter
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm 

# 1. Inisialisasi Stopword dan Stemmer (Sastrawi)
sw_factory = StopWordRemoverFactory()
stop_words = set(sw_factory.get_stop_words())

st_factory = StemmerFactory()
stemmer = st_factory.create_stemmer()

# Langkah 1 CLeaning Basic
def basic_cleaning(text):
    text = str(text).lower()
    text = text.replace('baca berita dengan sedikit iklan, klik di sini', '')
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Menjalankan Cleaning Dasar....")
df['temp_clean'] = df['Content'].swifter.apply(basic_cleaning)

# Langkah 2 Stemming
print("Membuat Dictionary Stemming....")
all_words = set("".join(df['temp_clean']).split())
unique_words = [w for w in all_words if w not in stop_words] 

word_map = {}
for word in tqdm(unique_words, desc="Stemming progpress"):
    word_map[word] = stemmer.stem(word)

# Langkah 3 Mapping
def procesing(text):
    tokens = text.split()
    stemmed_tokens = [word_map[w] for w in tokens if w in word_map]
    return stemmed_tokens

print("Menyusun Hasil Akhir.....")
df['cleaned_text'] = df['temp_clean'].swifter.apply(procesing)

# Hapus kolom
df.drop(columns=['temp_clean'], inplace=True)

# Simpan hasil
df.to_csv('dataset_processing.csv', index=False)
print("Selesai dan Tersimpan")


Menjalankan Cleaning Dasar....


Pandas Apply:   0%|          | 0/26753 [00:00<?, ?it/s]

Membuat Dictionary Stemming....


Stemming progpress: 100%|██████████| 78408/78408 [3:15:51<00:00,  6.67it/s]  


Menyusun Hasil Akhir.....


Pandas Apply:   0%|          | 0/26753 [00:00<?, ?it/s]

Selesai dan Tersimpan


#### Load Dataset Hasil Procesing

In [14]:
df_procesing = pd.read_csv('../data/dataset_processing.csv')
df_procesing['cleaned_text'].head()

0    ['gempa', 'bumi', 'kuat', 'magnitudo', 'm', 'g...
1    ['presiden', 'prabowo', 'subianto', 'temu', 'g...
2    ['jakarta', 'kompas', 'com', 'wakil', 'ketua',...
3    ['tangerang', 'selatan', 'kompas', 'com', 'ora...
4    ['jakarta', 'kompas', 'com', 'bakar', 'landa',...
Name: cleaned_text, dtype: object

In [16]:
from IPython.display import display, HTML

# Tampilkan tabel dengan lebar kolom tertentu
display(HTML(df[['cleaned_text']].head(10).to_html().replace('<td>', '<td style="max-width: 2500px; word-wrap: break-word;">')))

,cleaned_text
0,"[gempa, bumi, kuat, magnitudo, m, guncang, kabupaten, pulau, doi, malu, utara, gempa, ada, dalam, km, informasi, gempa, sampai, bmkg, akun, x, resmi, senin, sebut, gempa, jadi, pukul, wib, pusat, gempa, ada, laut, km, barat, laut, pulau, doi, tulis, bmkg, titik, gempa, ada, koordinat, lintang, utara, bujur, timur, bmkg, kata, informasi, gempa, beri, utama, cepat, jadi, ubah, data, gempa, rasa, skala, modified, mercalli, intensity, mmi, ii, iii, tagulandang, skala, ii, arti, getar, rasa, beberapa, orang, benda, benda, ringan, gantung, goyang, skala, iii, arti, getar, rasa, nyata, rumah, asa, getar, akan, truk, lalu]"
1,"[presiden, prabowo, subianto, temu, gubernur, banten, pilih, andra, soni, istana, hari, apa, bahas, momen, temu, abadi, unggah, akun, instagram, andra, soni, jumat, andra, kena, kemeja, lengan, panjang, putih, prabowo, pakai, baju, safari, cokelat, muda, dua, foto, latar, belakang, istana, merdeka, jakarta, andra, syukur, dapat, sempat, temu, prabowo, alhamdulillah, sempat, silaturahmi, presiden, republik, indonesia, bapak, prabowo, subianto, tulis, andra, terang, unggah, sebut, andra, doa, prabowo, selalu, beri, sehat, sukses, pimpin, negeri, moga, beliau, selalu, beri, kuat, sukses, pimpin, negeri, tuju, indonesia, maju, ujar, tahu, pasang, andra, soni, dimyati, natakusumah, dapat, suara, banyak, pilgub, banten, hasil, rekapitulasi, kpu, provinsi, banten, andra, dimyati, ...]"
2,"[jakarta, kompas, com, wakil, ketua, mpr, ri, eddy, soeparno, ingat, pentignya, mitigasi, kesiap, manajemen, krisis, baik, jadi, bencana, besar, sampai, eddy, respons, bencana, buat, rt, jakarta, camat, bekas, rendam, dalam, air, variasi, terus, terus, respons, bencana, jadi, perlu, langkah, mitigasi, kesiap, manajemen, krisis, lebih, baik, dampak, minimal, ujar, eddy, selasa, kutip, siar, pers, politikus, partai, amanat, nasional, kata, bencana, banjir, rupa, bukti, nyata, krisis, iklim, makin, ancam, pola, banjir, terus, ulang, perlu, hadap, strategi, lebih, sistematis, memitigasi, dampak, ubah, iklim, bukan, pertama, kali, hadap, banjir, besar, pola, terus, ulang, tahun, kalau, bijak, lebih, serius, depan, situasi, makin, buruk, ujar, eddy, ...]"
3,"[tangerang, selatan, kompas, com, orang, remaja, kemudi, mobil, toyota, hilux, tindak, tugas, satu, lalu, lintas, satlantas, polres, tangerang, selatan, dapat, guna, pelat, nomor, kendara, asing, tindak, laku, jalan, letnan, sutopo, serpong, tangerang, selatan, senin, sore, foto, video, tindak, remaja, unggah, akun, instagram, satlantaspolrestangsel, unggah, foto, video, lihat, pelat, mobil, guna, tulis, huruf, arab, rupa, pelat, timur, tengah, slide, foto, ikut, ungkap, mobil, sebut, milik, pelat, indonesia, nomor, d, fj, tunjuk, kendara, pribadi, wilayah, bandung, jawa, barat, kepala, bagi, operasional, kbo, satlantas, polres, tangerang, selatan, iptu, hery, sulistiono, jelas, tindak, laku, guna, pelat, nomor, sesuai, untuk, anggota, patroli, temu, suatu, kendara, ...]"
4,"[jakarta, kompas, com, bakar, landa, kawasan, mukim, padat, jalan, mangga, besar, xiii, lurah, mangga, selatan, camat, sawah, besar, jakarta, pusat, sebab, tiga, warga, dapat, tangan, medis, tiga, warga, muhamad, zan, tahun, alami, luka, sobek, jari, jempol, kelingking, nurlela, salman, alami, kondisi, lemas, akibat, panik, marsel, anggota, rawan, bakar, sesak, napas, papar, asap, tiga, korban, langsung, dapat, tangan, medis, tim, sehat, pasti, kondisi, tangan, baik, ujar, kepala, dinas, tanggulang, kebakardan, selamat, jakarta, satriadi, gunawan, kompas, com, sabtu, bakar, jadi, sabtu, sore, pukul, wib, jinak, pukul, wib, total, unit, mobil, orang, personel, madam, bakar, kerah, tangan, peristiwa, sebut, satriadi, kata, bakar, sebut, ...]"
5,"[jakarta, kompas, com, polres, metro, jakarta, barat, laku, olah, tempat, jadi, perkara, tkp, glodok, plaza, jakarta, barat, proses, bersih, selesai, kasatreskrim, polres, metro, jakarta, barat, akbp, arfan, sipayung, ungkap, olah, tkp, laksana, material, mate